In [6]:
import os, sys, pathlib
# Work from the repo root: find the folder containing `src/`, put it on the import path,
# and chdir into it so imports AND relative paths (configs/, data/, outputs/) resolve the
# same as running a script from the project root.
ROOT = pathlib.Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

# Drop any stale/namespace `src`/`scripts` cached by an earlier failed import.
for _m in [m for m in sys.modules if m == 'scripts' or m.startswith('scripts.')
           or m == 'src' or m.startswith('src.')]:
    del sys.modules[_m]

print('working dir:', ROOT)

working dir: e:\Putyaga\diplom\amc-hardware-domain-shift


In [7]:
from pathlib import Path
import numpy as np
from src.data import MODULATION_CLASSES

RUNS = Path("runs")
files = sorted(RUNS.glob("*/*/predictions.npz"))
print(len(files), "cells")
for f in files:
    print(" ", f.parent.relative_to(RUNS))

5 cells
  baseline\seed10
  baseline\seed11
  baseline\seed12
  baseline\seed13
  baseline\seed14


In [8]:
d = np.load(files[0])
pred, true, snr = d["pred"], d["true"], d["snr"]
print(pred.shape, true.shape, snr.shape)
print("класи в true:", np.unique(true).size, "з", len(MODULATION_CLASSES))
print("класи в pred:", np.unique(pred).size)
print("SNR:", np.unique(snr))

(35568,) (35568,) (35568,)
класи в true: 24 з 24
класи в pred: 23
SNR: [-20 -18 -16 -14 -12 -10  -8  -6  -4  -2   0   2   4   6   8  10  12  14
  16  18  20  22  24  26  28  30]


In [9]:
acc = (pred == true).mean()
mask = snr >= 0
print(f"усі кадри:   {acc:.4f}")
print(f"SNR >= 0 dB: {(pred[mask] == true[mask]).mean():.4f}")

усі кадри:   0.4387
SNR >= 0 dB: 0.6259


In [10]:
for s in np.unique(snr):
    m = snr == s
    print(f"{s:>4} dB  n={m.sum():>6}  acc={(pred[m] == true[m]).mean():.3f}")

 -20 dB  n=  1368  acc=0.039
 -18 dB  n=  1368  acc=0.042
 -16 dB  n=  1368  acc=0.045
 -14 dB  n=  1368  acc=0.050
 -12 dB  n=  1368  acc=0.060
 -10 dB  n=  1368  acc=0.106
  -8 dB  n=  1368  acc=0.156
  -6 dB  n=  1368  acc=0.221
  -4 dB  n=  1368  acc=0.292
  -2 dB  n=  1368  acc=0.381
   0 dB  n=  1368  acc=0.454
   2 dB  n=  1368  acc=0.525
   4 dB  n=  1368  acc=0.571
   6 dB  n=  1368  acc=0.635
   8 dB  n=  1368  acc=0.644
  10 dB  n=  1368  acc=0.652
  12 dB  n=  1368  acc=0.654
  14 dB  n=  1368  acc=0.652
  16 dB  n=  1368  acc=0.653
  18 dB  n=  1368  acc=0.648
  20 dB  n=  1368  acc=0.644
  22 dB  n=  1368  acc=0.657
  24 dB  n=  1368  acc=0.659
  26 dB  n=  1368  acc=0.650
  28 dB  n=  1368  acc=0.651
  30 dB  n=  1368  acc=0.664


In [11]:
for c in range(len(MODULATION_CLASSES)):
    m = mask & (true == c)
    if m.sum():
        print(f"{MODULATION_CLASSES[c]:>10}  n={m.sum():>5}  recall={(pred[m] == c).mean():.3f}")

       OOK  n=  912  recall=1.000
      4ASK  n=  912  recall=0.927
      8ASK  n=  912  recall=0.897
      BPSK  n=  912  recall=1.000
      QPSK  n=  912  recall=0.978
      8PSK  n=  912  recall=0.919
     16PSK  n=  912  recall=0.038
     32PSK  n=  912  recall=0.000
    16APSK  n=  912  recall=0.878
    32APSK  n=  912  recall=0.918
    64APSK  n=  912  recall=0.087
   128APSK  n=  912  recall=0.408
     16QAM  n=  912  recall=0.785
     32QAM  n=  912  recall=0.380
     64QAM  n=  912  recall=0.008
    128QAM  n=  912  recall=0.000
    256QAM  n=  912  recall=0.717
 AM-SSB-WC  n=  912  recall=0.970
 AM-SSB-SC  n=  912  recall=0.084
 AM-DSB-WC  n=  912  recall=0.448
 AM-DSB-SC  n=  912  recall=0.589
        FM  n=  912  recall=1.000
      GMSK  n=  912  recall=1.000
     OQPSK  n=  912  recall=0.989


In [12]:
for f in files:
    z = np.load(f)
    m = z["snr"] >= 0
    print(f.parent.name, f"{(z['pred'][m] == z['true'][m]).mean():.4f}")

seed10 0.6259
seed11 0.6412
seed12 0.6507
seed13 0.7259
seed14 0.6063
